# Path 3A — The BigQuery World: Python Exploration in `BigFrames` $\to$ In-Warehouse `BQML` & BigQuery AI (`@skill:bigframes-codegen`, `@skill:ml-best-practices`, `@skill:notebook-guidance`)

In reality, enterprise customers choose either the **BigQuery-native ML path (Path 3A)** or the **Spark + GEAP ML path (Path 3B)** rather than mixing both in the same modeling workflow. Both paths are sustained end-to-end by **DAK Notebooks in VS Code (`@skill:notebook-guidance`)**.

This notebook demonstrates **Path 3A (The BigQuery World)**:
1. **Exploratory Feature Engineering & Distributed Python Pushdown (`@skill:bigframes-codegen`)**: Use `bigframes.pandas` and `bigframes.ml` (`StandardScaler`, `XGBClassifier`) to explore features and prototype over millions of BigQuery rows with zero local memory bottlenecks.
2. **Chronological Train/Test Split (`@skill:ml-best-practices`)**: Split data by `event_timestamp` *prior* to fitting scalers or trees to eliminate temporal leakage.
3. **In-Warehouse `BQML` & BigQuery AI Execution (`alt_a_bqml_ai_sql/`)**: Directly from the same notebook session, trigger and inspect BigQuery AI (`AI.GENERATE_TABLE`, `VECTOR_SEARCH` RRF, `AI.FORECAST`) and production `BQML` (`BOOSTED_TREE_CLASSIFIER` with `DATA_SPLIT_METHOD = 'SEQ'` and counterfactual Rank-1 debiasing).

In [ ]:
import bigframes.pandas as bpd
from bigframes.ml.ensemble import XGBClassifier
from bigframes.ml.preprocessing import StandardScaler

PROJECT_ID = "your-gcp-project-id"
bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = "US"

impressions_bf = bpd.read_gbq(f"{PROJECT_ID}.raw_retail_core.search_impressions")
print("BigFrames DataFrame schema:", impressions_bf.dtypes)

### Step 2: Chronological Split & Distributed Feature Scaling (`@skill:ml-best-practices`)
We sort chronologically by `event_timestamp` and split the earliest 80% into `train_bf` and the most recent 20% into `test_bf` before fitting `StandardScaler`.

In [ ]:
FEATURE_COLS = [
    "query_length_tokens",
    "displayed_rank_position",
    "lexical_bm25_score",
    "semantic_cosine_similarity",
    "historical_item_ctr",
    "price_usd",
    "average_rating",
]
LABEL_COL = "purchased"

cutoff_ts = "2026-09-20T00:00:00Z"
train_bf = impressions_bf[impressions_bf["event_timestamp"] <= cutoff_ts]
test_bf = impressions_bf[impressions_bf["event_timestamp"] > cutoff_ts]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_bf[FEATURE_COLS])
y_train = train_bf[[LABEL_COL]]

X_test_scaled = scaler.transform(test_bf[FEATURE_COLS])
y_test = test_bf[[LABEL_COL]]

### Step 3: Train `XGBClassifier`, Evaluate `model.score()`, and Score Counterfactual Rank-1 Propensity
We train `XGBClassifier` in-engine, verify holdout accuracy via `model.score()`, and write debiased predictions back to `serving_clarity_intelligence.bigframes_reranked_results`.

In [ ]:
xgb_model = XGBClassifier(n_estimators=40, max_depth=5, learning_rate=0.15)
xgb_model.fit(X_train_scaled, y_train)

eval_metrics = xgb_model.score(X_test_scaled, y_test)
print("Holdout Evaluation Metrics (BigFrames in-warehouse):", eval_metrics)

# Counterfactual Rank-1 Position-Bias Debiasing
debiased_features_bf = test_bf[FEATURE_COLS].assign(displayed_rank_position=1)
debiased_preds_bf = xgb_model.predict(scaler.transform(debiased_features_bf))
debiased_preds_bf.to_gbq(
    f"{PROJECT_ID}.serving_clarity_intelligence.bigframes_reranked_results",
    if_exists="replace",
)

### Step 4: Handoff from `BigFrames` Exploration to Production `BQML` & BigQuery AI (`alt_a_bqml_ai_sql/`)
Once feature importance and chronological split thresholds are validated in `BigFrames`, teams in the **BigQuery World (Path 3A)** materialize the production `BOOSTED_TREE_CLASSIFIER` and counterfactual Rank-1 re-ranker directly inside BigQuery ML (`03_bqml_conversion_reranker.sql`) and score 30-day category demand via `AI.FORECAST` (`04_ai_forecast_demand.sql`)—all orchestrated from the same DAK Notebook via `bpd.read_gbq_query()`.

In [ ]:
# Inspect production BQML BOOSTED_TREE_CLASSIFIER evaluation & counterfactual Rank-1 lift directly via BigFrames
bqml_eval_df = bpd.read_gbq_query(
    f"""
    SELECT
      roc_auc,
      precision,
      recall,
      log_loss
    FROM ML.EVALUATE(
      MODEL `{PROJECT_ID}.modeled_search_warehouse.clarity_conversion_reranker_bdt`
    )
    """
)
print("Production BQML BOOSTED_TREE_CLASSIFIER Evaluation:")
print(bqml_eval_df.head(5))

bqml_reranked_summary_df = bpd.read_gbq_query(
    f"""
    SELECT
      query_tier,
      COUNT(*) AS scored_impressions,
      ROUND(AVG(debiased_conversion_probability), 4) AS avg_debiased_cvr_prob
    FROM `{PROJECT_ID}.serving_clarity_intelligence.clarity_reranked_search_results`
    GROUP BY query_tier
    """
)
print(bqml_reranked_summary_df.head(10))